Contexto de la tarea:

Imagina que estás desarrollando el sistema interno de asignación de pedidos para una
aplicación de entrega de comida a domicilio.

1. Flujo Ordinario (La Cola - FIFO): Durante el día, los pedidos de los clientes entran en
una Cola. El primer cliente que pide es el primero cuyo plato se cocina y se envía. El
flujo es ordenado.

2. Flujo de Cancelaciones/Reclamos (La Pila - LIFO): De repente, llega la Hora Pico (lluvia,
fin de semana). Los repartidores empiezan a demorarse y el sistema recibe
notificaciones de Emergencia/Reclamo (como por ejemplo: "Mi pedido llegó frío",
"Falta la bebida"). Estos reclamos no pueden esperar al final de la cola; se apilan en
3

una Pila de atención prioritaria. El último reclamo en entrar es el primero que el
supervisor humano debe resolver de inmediato para evitar perder al cliente.

El Problema de la app:

La aplicación se satura por completo. El gerente decide activar el "Modo de
Contingencia Masiva". Para no perder el rastro de los pedidos comunes que quedaron
atrapados en la cola y que ya están retrasados, el sistema debe congelar la cola y
transferir todos esos pedidos pendientes a la pila de reclamos. Al hacer esto, el sistema
procesará primero los pedidos ordinarios más recientes (los que tienen más probabilidad
de salvarse antes de que se cancelen), invirtiendo el orden de atención.


Instrucciones:

Parte 1. Estructura Base Obligatoria

Copia exactamente este código base en tu archivo de desarrollo. No se permite
modificar los atributos existentes:


In [ ]:
class NodoPedido:

    def __init__(self, id_pedido: int, tipo: str, descripcion: str):

        self.id_pedido = id_pedido # Ejemplo: 501, 502

        self.tipo = tipo # Únicamente: "ESTÁNDAR" o "RECLAMO"

        self.descripcion = descripcion # Ej: "1 Hamburguesa con papas"

        self.siguiente = None # Puntero al siguiente Nodo en memoria

class GestorPedidos:

    def __init__(self):
 # Punteros para la Cola de Pedidos Estándar (FIFO)

        self.cola_frente = None

        self.cola_fin = None

 # Puntero para la Pila de Reclamos Urgentes (LIFO)
        self.pila_tope = None

#PARTE 2
#REGISTRAR PEDIDOS
    def registrar_pedido(self, id_pedido: int, tipo: str, descripcion: str):

        nuevo = NodoPedido(id_pedido, tipo, descripcion)

        # PEDIDOS ESTÁNDAR = COLA
        if tipo == "ESTÁNDAR":

            # Si la cola está vacía
            if self.cola_frente is None:
                self.cola_frente = nuevo
                self.cola_fin = nuevo

            else:
                self.cola_fin.siguiente = nuevo
                self.cola_fin = nuevo

        # RECLAMOS = PILA
        elif tipo == "RECLAMO":

            nuevo.siguiente = self.pila_tope
            self.pila_tope = nuevo

# DESPACHAR PEDIDOS
    def despachar_siguiente(self):

        # PRIORIDAD A RECLAMOS
        if self.pila_tope is not None:

            atendido = self.pila_tope
            self.pila_tope = self.pila_tope.siguiente

            atendido.siguiente = None
            return atendido

        # SI NO HAY RECLAMOS
        elif self.cola_frente is not None:

            atendido = self.cola_frente
            self.cola_frente = self.cola_frente.siguiente

            # Si la cola quedó vacía
            if self.cola_frente is None:
                self.cola_fin = None

            atendido.siguiente = None
            return atendido

        # NO HAY NADA
        return None
    
#ACTIVAR CONTINGENCIA MASIVA
    
    def activar_contingencia_masiva(self):

        while self.cola_frente is not None:

            # Sacar del frente de la cola
            actual = self.cola_frente
            self.cola_frente = self.cola_frente.siguiente

            # Apilar en la pila
            actual.siguiente = self.pila_tope
            self.pila_tope = actual

        # La cola queda vacía
        self.cola_fin = None

In [12]:
gestor = GestorPedidos()

cantidad = int(input("¿Cuántos pedidos desea registrar?: "))

for i in range(cantidad):

    print(f"\nPedido #{i+1}")

    id_pedido = int(input("Ingrese ID del pedido: "))
    tipo = input("Ingrese tipo (ESTÁNDAR o RECLAMO): ").upper()
    descripcion = input("Ingrese descripción: ")

    gestor.registrar_pedido(id_pedido, tipo, descripcion)


    print("\nPedido registrado:")
    print(f"ID: {id_pedido}")
    print(f"Tipo: {tipo}")
    print(f"Descripción: {descripcion}")

print("\n--- DESPACHANDO PEDIDOS ---")

while True:

    pedido = gestor.despachar_siguiente()

    if pedido is None:
        print("No hay más pedidos.")
        break

    print(f"Atendiendo pedido {pedido.id_pedido} - {pedido.tipo}")
    print(f"Descripción: {pedido.descripcion}")


Pedido #1

Pedido registrado:
ID: 1
Tipo: RECLAMO
Descripción: Comida

Pedido #2

Pedido registrado:
ID: 2
Tipo: ESTANDAR
Descripción: Laptop

Pedido #3

Pedido registrado:
ID: 3
Tipo: RECLAMO
Descripción: Iphone

--- DESPACHANDO PEDIDOS ---
Atendiendo pedido 3 - RECLAMO
Descripción: Iphone
Atendiendo pedido 1 - RECLAMO
Descripción: Comida
No hay más pedidos.


Parte 3. Actividad de Reflexión: El Caso del Puntero Fantasma

Un desarrollador novato intentó implementar el método despachar_siguiente() para el
GestorPedidos, pero cometió un error crítico de lógica.

A continuación se muestra su código:

def despachar_siguiente(self):

 //Intento de dequeue en la cola ordinaria cuando la pila está vacía

    if self.pila_tope is None and self.cola_frente is not None:

        pedido_despachado = self.cola_frente

        self.cola_frente = self.cola_frente.siguiente

    return pedido_despachad

1. El código de arriba funciona bien la mayoría de las veces, pero provoca un "bug"

grave (efecto colateral) cuando en la cola queda un solo elemento. Explica

detalladamente qué pasa con el puntero self.cola_fin en ese caso específico si se

ejecuta este código.

Lo que sucede el error es que cuando la cola tiene un solo pedido y ese pedido se elimina, la cola termina vacia.El problema llega que el programa solo actualiza el inicio de la cola (cola_frente), pero no va a actualizar el final (cola_fin). Entonces pareciera que la cola estuviera vacia pero todavia tiene una referencia al ultimo nodo que eliminamos.Esto causa errores cuando intentamos agregar un nuevo pedido, porque el sistema sigue conectado a un nodo que ya no existe.


2. ¿Qué consecuencias tendría este error la próxima vez que un cliente intente registrar

un pedido estándar usando registrar_pedido()? Justifica tu respuesta dibujando el

comportamiento de la memoria. Utiliza cualquier herramienta digital para el dibujo

(no imágenes hechas en IA, dibujo de la creatividad de cada uno).


La proxima vez que un cliente ingrese un pedido, el sistema se va a comportar de manera incorrecta porque, cola_fin esta apuntando al nodo incorrecto porque aunque parece que la cola esta vacia, todavia esta existiendo una referencia al ultimo nodo anterior. Entonces, cuando agregemos un nuevo pedido, este pedido se va a enlazar al nodo viejo en lugar de iniciar una nueva cola, ocasionando que tengamos una estructura corrupta, perdida de pedidos o conexiones incorrectas entre los nodos.